In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/raesiow/cleaned-off/off_filtered_all_countries.csv
/kaggle/input/datasets/raesiow/catalogue-predictions/catalogue_predictions.csv
/kaggle/input/datasets/raesiow/gi-dataset/atkinson_merged.csv
/kaggle/input/datasets/raesiow/gi-dataset/atkinson_off_fuzzy_matches.csv
/kaggle/input/datasets/raesiow/gi-dataset/gi_table_folds.csv


In [2]:
# paths found by the path finder
catalogue_path = "/kaggle/input/datasets/raesiow/catalogue-predictions/catalogue_predictions.csv"  # merged catalogue file
off_all_path = "/kaggle/input/datasets/raesiow/cleaned-off/off_filtered_all_countries.csv"  # OFF, all countries

# load the catalogue, keeping the barcode as text
catalogue = pd.read_csv(catalogue_path, dtype={"code": str})
# load all-countries OFF, keeping the barcode as text
off_all = pd.read_csv(off_all_path, dtype={"code": str})

# show the sizes of both tables
print("catalogue rows:", len(catalogue))
print("off_all rows:", len(off_all))

# check that the barcodes are unique in both tables
print("duplicate codes in catalogue:", catalogue["code"].duplicated().sum())
print("duplicate codes in off_all:", off_all["code"].duplicated().sum())

# keep only the nutrient columns we need, with the barcode as the row label
nutrient_cols = ["carb_excl_fiber_g", "sugars_g", "fiber_g", "fat_g", "protein_g"]  # the 5 GI features
off_nutrients = off_all.set_index("code")[nutrient_cols]  # barcode -> 5 nutrients

# look up the catalogue barcodes, in catalogue order (missing barcodes become NaN)
catalogue_nutrients = off_nutrients.reindex(catalogue["code"])  # one row per catalogue product

# count catalogue products that were NOT found in the all-countries file
print("catalogue products not found:", catalogue_nutrients["carb_excl_fiber_g"].isna().sum())

# compare carbs with the value stored in the catalogue file (should be the same)
carb_difference = catalogue_nutrients["carb_excl_fiber_g"].values - catalogue["carb_excl_fiber_g"].values  # new minus old
# show how many products differ by more than a tiny amount
print("carb values that differ:", (np.abs(carb_difference) > 0.01).sum())

catalogue rows: 71298
off_all rows: 376917
duplicate codes in catalogue: 0
duplicate codes in off_all: 0
catalogue products not found: 0
carb values that differ: 0


In [3]:
# path of the GI table with the 6,851 products and the 5 folds
gi_path = "/kaggle/input/datasets/raesiow/gi-dataset/gi_table_folds.csv"  # GI products
# load it, keeping the barcode as text
gi_table = pd.read_csv(gi_path, dtype={"code": str})  # one row per GI product

# the 5 nutrient columns used as features (same list as before)
nutrient_cols = ["carb_excl_fiber_g", "sugars_g", "fiber_g", "fat_g", "protein_g"]  # 5 GI features

# add pnns_groups_2 to the GI products by looking up each barcode in all-countries OFF
gi_table["pnns_groups_2"] = off_all.set_index("code")["pnns_groups_2"].reindex(gi_table["code"]).values  # category per product
# replace missing categories by the word "unknown" so they count as one group
gi_table["pnns_groups_2"] = gi_table["pnns_groups_2"].fillna("unknown")  # no empty values left
# do the same for the catalogue
catalogue["pnns_groups_2"] = catalogue["pnns_groups_2"].fillna("unknown")  # no empty values left

# turn GI numbers into bands with the same cut points as before: low <= 55, medium 56-69, high >= 70
gi_table["gi_band"] = pd.cut(gi_table["gi"], bins=[-np.inf, 55, 69, np.inf], labels=["low", "medium", "high"])  # band per product
# convert the band names to numbers 0, 1, 2 for the classifier
gi_table["gi_band_number"] = gi_table["gi_band"].map({"low": 0, "medium": 1, "high": 2}).astype(int)  # 0 = low, 2 = high

# log1p version of the 5 nutrients for the GI products
gi_features_log = np.log1p(gi_table[nutrient_cols])  # training features
# log1p version of the 5 nutrients for the catalogue (same barcode order as catalogue)
catalogue_features_log = np.log1p(catalogue_nutrients[nutrient_cols].reset_index(drop=True))  # catalogue features

# the list of categories seen in the GI products (these become the one-hot columns)
category_list = sorted(gi_table["pnns_groups_2"].unique())  # categories the model can learn
# one-hot for GI products: one 0/1 column per category
gi_category = pd.get_dummies(gi_table["pnns_groups_2"]).reindex(columns=category_list, fill_value=0).astype(int)  # 0/1 table
# one-hot for the catalogue, using the SAME columns; unseen categories get all zeros
catalogue_category = pd.get_dummies(catalogue["pnns_groups_2"]).reindex(columns=category_list, fill_value=0).astype(int)  # 0/1 table

# checks
print("GI products:", len(gi_table))  # expect 6851
print("number of categories in GI:", len(category_list))  # expect about 37
print("band counts:")  # heading
print(gi_table["gi_band"].value_counts())  # how many low / medium / high products
print("sum of food weights:", round(gi_table["food_weight"].sum(), 1))  # expect about 629
print("missing values in GI features:", gi_features_log.isna().sum().sum())  # expect 0
print("missing values in catalogue features:", catalogue_features_log.isna().sum().sum())  # expect 0
print("smallest catalogue nutrient value:", catalogue_nutrients[nutrient_cols].min().min())  # expect 0 or more
print("catalogue products with a category the GI data never saw:", (~catalogue["pnns_groups_2"].isin(category_list)).sum())  # these get all zeros

GI products: 6851
number of categories in GI: 37
band counts:
gi_band
low       3945
medium    1785
high      1121
Name: count, dtype: int64
sum of food weights: 629.0
missing values in GI features: 0
missing values in catalogue features: 0
smallest catalogue nutrient value: -0.0070496323529431
catalogue products with a category the GI data never saw: 1476


In [4]:
# count negative values in each catalogue nutrient column
print("negative values per column:")  # heading
print((catalogue_nutrients[nutrient_cols] < 0).sum())  # how many below zero in each column

# do the same check for the GI products
print("negative values in GI products:", (gi_table[nutrient_cols] < 0).sum().sum())  # expect 0

# set any negative catalogue nutrient to 0
catalogue_nutrients_clipped = catalogue_nutrients[nutrient_cols].clip(lower=0)  # nothing below zero now
# rebuild the log1p catalogue features from the clipped values
catalogue_features_log = np.log1p(catalogue_nutrients_clipped.reset_index(drop=True))  # same barcode order as catalogue

# smallest value after clipping (should be 0)
print("smallest value after clipping:", catalogue_nutrients_clipped.min().min())  # expect 0.0

# find the catalogue products whose category the GI data never saw
unseen_mask = ~catalogue["pnns_groups_2"].isin(category_list)  # True for unseen categories
# count them by category name, largest first
print("unseen categories in the catalogue:")  # heading
print(catalogue.loc[unseen_mask, "pnns_groups_2"].value_counts().head(15))  # top 15 unseen categories

negative values per column:
carb_excl_fiber_g    0
sugars_g             0
fiber_g              1
fat_g                0
protein_g            0
dtype: int64
negative values in GI products: 0
smallest value after clipping: 0.0
unseen categories in the catalogue:
pnns_groups_2
Processed meat    1453
Offals              14
Fruit nectars        9
Name: count, dtype: int64


In [5]:
# the catalogue products whose category the GI data never saw
unseen_products = catalogue[unseen_mask]  # 1,476 rows

# how many of them already have the low-carb flag (carbs under 5 g per 100 g)
print("unseen and low carb:", unseen_products["flag_low_carb"].sum())  # these are already "GI not estimated"
# how many of them already have the sauce flag
print("unseen and sauce:", unseen_products["flag_sauce"].sum())  # sauces are also not estimated
# how many have a GI number given in the old catalogue file
print("unseen with an old gi_pred:", unseen_products["gi_pred"].notna().sum())  # these still show a number
# how many of the unseen products were NOT flagged by either rule
print("unseen and not flagged:", ((~unseen_products["flag_low_carb"]) & (~unseen_products["flag_sauce"])).sum())  # these keep a prediction

unseen and low carb: 1098
unseen and sauce: 0
unseen with an old gi_pred: 1476
unseen and not flagged: 378


In [6]:
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor  # the two GBM types

# ---------- (a) band classifier 0: 5 nutrients only ----------
x_train_nut = gi_features_log.values  # 6,851 x 5 training features
x_catalogue_nut = catalogue_features_log.values  # 71,298 x 5 catalogue features
y_band_all = gi_table["gi_band_number"].values  # bands as numbers: 0 low, 1 medium, 2 high
food_w = gi_table["food_weight"].values  # each food counts once

# total food weight in each band
class_total = np.array([food_w[y_band_all == c].sum() for c in range(3)])  # one total per band
# balanced class weight: rare bands count more
class_w = class_total.sum() / (3 * class_total)  # same formula as notebook 18
# food weight x class weight for every training product
sample_w = food_w * class_w[y_band_all]  # one weight per product

# fresh classifier with the settings fixed before running
band_model = HistGradientBoostingClassifier(max_iter=200, learning_rate=0.05, random_state=42)  # same as 18
# train on all 6,851 products
band_model.fit(x_train_nut, y_band_all, sample_weight=sample_w)  # one final model
# predict a band number for every catalogue product
catalogue_band_number = band_model.predict(x_catalogue_nut)  # 0, 1 or 2

# ---------- (b) regression A: 5 nutrients + category ----------
x_train_A = np.hstack([x_train_nut, gi_category.values])  # 5 nutrients + 37 category columns
x_catalogue_A = np.hstack([x_catalogue_nut, catalogue_category.values])  # same columns for the catalogue

# fresh regressor with the same settings
number_model = HistGradientBoostingRegressor(max_iter=200, learning_rate=0.05, random_state=42)  # GBM regressor
# train on the GI numbers, each food counting once
number_model.fit(x_train_A, gi_table["gi"].values, sample_weight=food_w)  # one final model
# predict an approximate GI number for every catalogue product
catalogue_number = number_model.predict(x_catalogue_A)  # one number per product

# ---------- checks ----------
band_names = np.array(["low", "medium", "high"])  # names for 0, 1, 2
print("class weights (low, medium, high):", class_w.round(2))  # expect high largest
print("new band counts:")  # heading
print(pd.Series(band_names[catalogue_band_number]).value_counts())  # bands in the catalogue
print("new number: min", catalogue_number.min().round(1), "mean", catalogue_number.mean().round(1), "max", catalogue_number.max().round(1))  # range of the numbers
# band you would get from the number, using the same cut points
band_from_number = pd.cut(catalogue_number, bins=[-np.inf, 55, 69, np.inf], labels=[0, 1, 2]).astype(int)  # 0, 1, 2
print("band and number disagree:", (band_from_number != catalogue_band_number).sum(), "of", len(catalogue))  # count of disagreements

class weights (low, medium, high): [0.61 1.26 1.73]
new band counts:
low       39943
medium    20027
high      11328
Name: count, dtype: int64
new number: min -1.6 mean 51.8 max 100.8
band and number disagree: 22091 of 71298


In [7]:
# save the new results as columns in the catalogue table
catalogue["gi_band_new"] = band_names[catalogue_band_number]  # band from the classifier
catalogue["gi_number_new"] = np.clip(catalogue_number, 0, 100)  # number kept between 0 and 100
catalogue["gi_band_from_number"] = band_names[band_from_number]  # band you would get from the number

# products where the app will actually show a GI (not flagged as "not estimated")
shown = ~catalogue["gi_not_estimated"].astype(str).str.lower().eq("true")  # True = GI is shown
print("products with a GI shown:", shown.sum())  # expect about 53,338

# how often the band and the number disagree among those products
disagree_shown = (catalogue.loc[shown, "gi_band_new"] != catalogue.loc[shown, "gi_band_from_number"]).sum()  # count
print("disagree among shown:", disagree_shown, "(", round(100 * disagree_shown / shown.sum(), 1), "%)")  # percent

# table: classifier band (rows) against the band from the number (columns)
print(pd.crosstab(catalogue.loc[shown, "gi_band_new"], catalogue.loc[shown, "gi_band_from_number"]))  # how they disagree

# table: old band (rows) against new classifier band (columns)
print(pd.crosstab(catalogue.loc[shown, "gi_band"], catalogue.loc[shown, "gi_band_new"]))  # what changed from the old file

products with a GI shown: 53338
disagree among shown: 15879 ( 29.8 %)
gi_band_from_number  high    low  medium
gi_band_new                             
high                 2400    804    5334
low                   136  23661    4710
medium                766   4129   11398
gi_band_new  high    low  medium
gi_band                         
high         2190    130     798
low           753  24310    4168
medium       5595   4067   11327


In [8]:
# search words for the products we want to look at
search_words = ["white bread", "cornflakes", "rice cakes", "instant mashed potato", "potato salad"]  # five checks

# columns to show
show_cols = ["product_name", "pnns_groups_2", "carb_excl_fiber_g", "gi_band", "gi_band_new", "gi_number_new", "gi_not_estimated"]  # old band, new band, new number

# look at each search word in turn
for word in search_words:  # loop over the five words
    # catalogue products whose name contains the word (ignoring capital letters)
    found = catalogue[catalogue["product_name"].str.lower().str.contains(word, na=False)]  # matching rows
    print("-----", word, "| products found:", len(found))  # heading
    # show up to 5 random ones; the seed is fixed so we are not picking favourites
    print(found.sample(min(5, len(found)), random_state=42)[show_cols].round(1).to_string())  # table

----- white bread | products found: 168
                                   product_name pnns_groups_2  carb_excl_fiber_g gi_band gi_band_new  gi_number_new  gi_not_estimated
51683                               WHITE BREAD         Bread               45.6     low        high           48.9             False
14978                               White Bread         Bread               49.2    high      medium           68.9             False
39531                M&S Wildfarmed White Bread         Bread               48.4  medium      medium           59.1             False
14726                               White Bread         Bread               47.3  medium      medium           64.6             False
53181  Lightly Seeded Medium Sliced White Bread         Bread               40.3     low        high           53.8             False
----- cornflakes | products found: 16
                         product_name      pnns_groups_2  carb_excl_fiber_g gi_band gi_band_new  gi_number_new  gi_not

In [9]:
# load the merged Atkinson table (2,590 foods)
atk_path = "/kaggle/input/datasets/raesiow/gi-dataset/atkinson_merged.csv"  # merged GI table
atk = pd.read_csv(atk_path)  # one row per merged food

# find foods with "rice cake" in the name (merge_key is the sorted words of the name)
rice_rows = atk[atk["merge_key"].str.contains("cake", na=False) & atk["merge_key"].str.contains("rice", na=False)]  # rice cake foods
# show the name and the GI values
print(rice_rows[["merge_key", "gi_median", "gi_min", "gi_max"]].to_string())  # real measured GI

                                                                     merge_key  gi_median  gi_min  gi_max
124                               1 algae cake dried g glutinous rice sea with       83.0    83.0    83.0
767                                ball cake cut glutinous glutinous rice with       48.0    48.0    48.0
770                                       balls balls cake glutinous rice rice       97.0    97.0    97.0
1310                                             brown cake prepared rice with       67.0    67.0    67.0
1376  cake cake chinese four or pan-fried plan rice shredded taro with yam yam       86.0    86.0    86.0
1384          cake glutinous pan-fried pastry pink rice rice rice stuffed with       97.0    97.0    97.0
1389                                                                 cake rice       82.0    82.0    82.0
1390                                            cakes calrose puffed rice rice       91.0    91.0    91.0
1391                                        ca

In [10]:
# barcodes of the products the final models were trained on
training_codes = set(gi_table["code"])  # 6,851 barcodes

# mark catalogue products that were in the GI training data
catalogue["in_gi_training"] = catalogue["code"].isin(training_codes)  # True = model has seen its GI label
print("catalogue products in GI training:", catalogue["in_gi_training"].sum(), "of", len(catalogue))  # overlap count

# the same count for products with "rice cake" in the name
rice_mask = catalogue["product_name"].str.lower().str.contains("rice cake", na=False)  # rice cake products
print("rice cake products:", rice_mask.sum(), "| of those in GI training:", (rice_mask & catalogue["in_gi_training"]).sum())  # overlap for rice cakes

catalogue products in GI training: 4032 of 71298
rice cake products: 150 | of those in GI training: 14


In [11]:
# keep the old band under a clear name
catalogue = catalogue.rename(columns={"gi_band": "gi_band_old"})  # old predicted band

# save the full catalogue with old and new GI columns (includes in_gi_training)
catalogue.to_csv("/kaggle/working/catalogue_predictions_v2.csv", index=False)  # new merged file
print("saved rows:", len(catalogue), "columns:", len(catalogue.columns))  # expect 71298 rows

saved rows: 71298 columns: 17
